In [1]:
%load_ext autoreload
%autoreload 2
import os
import sys
import trimesh
import fvdb
import argparse
from pprint import pprint
import tqdm
import h5py
from skimage import measure
from meshplot import plot
from scipy.spatial.distance import cdist
import joblib
import numpy as np
from scipy.spatial import KDTree
import igl
import fvdb
import matplotlib.pyplot as plt
import fvdb.nn as fvnn
import torch


# Import ssu packages
sys.path.append('../src')
sys.path.append('../config')
# config packages
import read_config
# src packages
import eval
import utils
import models
from logger import wandb_logging
from data_loader import ABC_dataset_loader
from utils import fvdb_utils as fu
from utils import ssu_tools as st 
from utils import mesh_tools as mt

In [2]:
def fetch_numpy_values(grid: fvdb.GridBatch, arr: np.array, size:int):
    '''fetches values from a numpy array based on the ijk indices in the grid'''
    ijk = grid.ijk.jdata.cpu().detach().numpy()+(size-1)//2
    
    if max(ijk[:, 0]) >= arr.shape[0] or max(ijk[:, 1]) >= arr.shape[1] or max(ijk[:, 2]) >= arr.shape[2]:
        # If indices are out of bounds, we can add the maximum value to the indices
        ijk = np.clip(ijk, 0, np.array(arr.shape) - 1)
        # print(f"Indices out of bounds. Clipping to max shape: {arr.shape}")
    
    values = arr[ijk[:, 0], ijk[:, 1], ijk[:, 2]]
    return torch.tensor(values, dtype=torch.float32, device=grid.device)


In [3]:
def remove_small_from_big(big_torch, small_torch):
    """
    Remove rows from big_torch that exist in small_torch
    Both tensors should have shape (N, 3)
    """
    # Convert rows to a single value for comparison by creating a hash-like representation
    big_combined = big_torch[:, 0] * 1e14 + big_torch[:, 1] * 1e7 + big_torch[:, 2]
    small_combined = small_torch[:, 0] * 1e14 + small_torch[:, 1] * 1e7 + small_torch[:, 2]

    # Find which rows in big_torch are NOT in small_torch
    mask = ~torch.isin(big_combined, small_combined)
    
    return big_torch[mask]

In [4]:
def custom_subdivide_grid(grid: fvdb.GridBatch):
    '''custom subdivision of a grid to create a finer grid:
        [0,    1,    2] -->
        [0, 1, 2, 3, 4]'''
    all_ijk        = grid.ijk
    enabled_ijk    = grid.ijk_enabled.jdata
    disabled_mask  = grid.disabled_mask.jdata
    disabled_ijk   = all_ijk.rmask(disabled_mask).jdata

    m3g = torch.tensor(mt.mesh_grid(3),device=grid.device)-1
    fine = (2*all_ijk.jdata[:, None, :] + m3g[None, :, :]).view(-1, 3)
    # enabled_fine = (2*enabled_ijk[:, None, :]+ m3g[None, :, :]).view(-1, 3)
    disabled_fine = (2*disabled_ijk[:, None, :]+ m3g[None, :, :]).view(-1, 3)
    
    # print('grid shape:', grid.ijk.jdata.shape)
    # print('enabled_fine', enabled_fine.shape)
    # print('disabled_fine', disabled_fine.shape)

    # combined = torch.cat([enabled_fine, disabled_fine], dim=0)
    # combined_jagged = fvdb.JaggedTensor([combined])

    subdivide_grid = fvdb.gridbatch_from_ijk(fvdb.JaggedTensor(fine), 
                                   origins=grid.origins, 
                                   voxel_sizes=grid.voxel_sizes/2,
                                   mutable=True)
    disabled_child_jagged = fvdb.JaggedTensor([disabled_fine])
    subdivide_grid.disable_ijk(disabled_child_jagged)

    disabled_grid = fvdb.gridbatch_from_ijk(disabled_fine, 
                                            voxel_sizes=grid.voxel_sizes, 
                                            origins=grid.origins, mutable=True)
    # print('subdivide_grid shape:', subdivide_grid.ijk.jdata.shape)
    # subdivide_grid = subdivide_grid.contiguous()
    # remove the disabled mask from the subdivided grid
    # enabled_fine = remove_small_from_big(fine, disabled_fine)
    enabled_fine_jagged = subdivide_grid.ijk_enabled
    enabled_grid = fvdb.gridbatch_from_ijk(enabled_fine_jagged, 
                                   origins=grid.origins, 
                                   voxel_sizes=grid.voxel_sizes/2, 
                                   mutable=True)

    print('intial disable shape:', disabled_ijk.shape)
    print('subdivide_grid shape:', subdivide_grid.ijk.jdata.shape)
    print('enabled_grid shape:', enabled_grid.ijk.jdata.shape)
    print('disabled_grid shape:', disabled_grid.ijk.jdata.shape)
    return enabled_grid, disabled_grid


In [5]:
def scaled_sdf(sdf_arr: np.array, threshold: float):
    '''scales the SDF array by the threshold value'''
    return (threshold-1)*sdf_arr[:, None]

In [6]:
def trilinear(big_vdb_grid, small_vdb, predict=False):
    # copy and enables all ijk
    # small_vdb = small_vdb_i.copy()
    if not predict:
        new_centers = big_vdb_grid.grid_to_world(big_vdb_grid.ijk.float())
        up_feat = small_vdb.grid.sample_trilinear(new_centers, small_vdb.jdata)
    else:
        new_centers = big_vdb_grid.grid_to_world(big_vdb_grid.ijk.float())
        up_feat = small_vdb.grid.sample_trilinear(new_centers, small_vdb.jdata)
    return up_feat

In [7]:
def get_big_vdb_grid(small_vdb: fvnn.VDBTensor, sdf:np.ndarray, size: int, is_trilinear=False):
    # convert large sdf to vdb
    big_vdb_grid_e, big_vdb_grid_d = custom_subdivide_grid(small_vdb.grid)
    # small_vdb.grid.enable_ijk(small_vdb.grid.ijk.jdata)

    sdf_values = fetch_numpy_values(big_vdb_grid_e, sdf, 2*size-1)

    sdf_values = scaled_sdf(sdf_values, 65)

    large_vdb_e = fvnn.VDBTensor(big_vdb_grid_e, 
                                big_vdb_grid_e.jagged_like(sdf_values))
    
    # replace the disabled values with trilinear interpolation
    if is_trilinear:
        up_feat = trilinear(big_vdb_grid_d, small_vdb, predict=True)
        large_vdb_d = fvnn.VDBTensor(big_vdb_grid_d, up_feat)

    # linear interpolation on enabled indices
    up_feat = trilinear(big_vdb_grid_e, small_vdb)

    # large_vdb_upfeat = fvnn.VDBTensor(big_vdb_grid, up_feat)

    # mask values larger than 0.0001
    print('large_vdb.jdata', large_vdb_e.jdata.shape)
    print('up_feat.jdata', up_feat.jdata.shape)
    sub_feat = up_feat.jdata - large_vdb_e.jdata
    sub_feat = torch.abs(sub_feat) 
    vdb_mask = sub_feat <= 0.00001
    vdb_mask = vdb_mask.squeeze(-1)  # remove the last dimension if it exists

    large_vdb_e.grid.disable_ijk(large_vdb_e.grid.ijk.rmask(vdb_mask))
    # diable all large_vdb_d
    if is_trilinear:
        large_vdb_d.grid.disable_ijk(large_vdb_d.grid.ijk)
        # large_vdb = fvdb.jcat([large_vdb_e, large_vdb_d], dim=0)
        combined_coords = fvdb.JaggedTensor([
                    torch.cat([large_vdb_e.ijk.jdata, large_vdb_d.ijk.jdata], dim=0)
                ])
        combined_feats = torch.cat([large_vdb_e.jdata, large_vdb_d.jdata], dim=0)
        merged_grid = fvdb.gridbatch_from_ijk(
                                    combined_coords,
                                    voxel_sizes=large_vdb_e.grid.voxel_sizes,
                                    origins=large_vdb_e.grid.origins,
                                    mutable=True
                                )
        large_vdb = fvnn.VDBTensor(merged_grid, merged_grid.jagged_like(combined_feats))
    else:
        large_vdb = large_vdb_e
    print('#'*20)
    print('large_vdb shape:', large_vdb.jdata.shape)
    print('large_vdb_e shape:', large_vdb_e.grid.ijk_enabled.jdata.shape)
    # print('large_vdb_d shape:', large_vdb_d.jdata.shape)
    print('sum of masks:', torch.sum(vdb_mask).item())

    # if is_trilinear:
    #     up_feat = trilinear, large_vdb)
    #     large_vdb = fvnn.VDBTensor(big_vdb_grid, up_feat)
    return large_vdb, vdb_mask

In [8]:
def exp(file_name):
    # device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    device = 'cpu'

    # load sdf from hdf5 file
    _dir = '/data/workspaces/spanwar/dataset/preprocessing_nmc_data/data_preprocessing/get_groundtruth_NMC/gt_large'
    file = os.path.join(_dir, file_name)
    with h5py.File(file, 'r') as h5_file:
        sdf_32 = h5_file[f'{32}_sdf'][:]
        sdf_64 = h5_file[f'{64}_sdf'][:]
        sdf_128 = h5_file[f'{128}_sdf'][:]
    print('max sdf_32:', sdf_32.max(), 'min sdf_32:', sdf_32.min())
    print('max sdf_64:', sdf_64.max(), 'min sdf_64:', sdf_64.min())

    # create fvdb object
    size = 33
    ijk_mesh_grid = mt.mesh_grid(size)
    ijk_mesh_grid = ijk_mesh_grid.reshape(size, size, size, 3)
    
    # consider only the points where the mask is True
    # normalize the ijk coordinates to be centered around (0, 0, 0)
    mask = mt.make_mask_close(sdf_32, 65)  # 65 is the mask threshold
    ijk = torch.tensor(ijk_mesh_grid[mask], 
                        dtype=torch.int, 
                        device=device)-(size-1)//2
    grid = fvdb.gridbatch_from_ijk(fvdb.JaggedTensor(ijk), 
                                    voxel_sizes=(1/(size-1)), 
                                    origins=torch.tensor([0, 0, 0], 
                                    device=device),
                                    mutable=True)
    
    sdf_values = fetch_numpy_values(grid, sdf_32, size)
    sdf_values = scaled_sdf(sdf_values, 65)

    vdb_32 = fvnn.VDBTensor(grid, 
                                grid.jagged_like(sdf_values))


    vdb_64, vdb_64_mask = get_big_vdb_grid(vdb_32, sdf_64, 33)

    vdb_128, vdb_128_mask = get_big_vdb_grid(vdb_64, sdf_128, 65, is_trilinear=True)
    return vdb_32, vdb_64, vdb_64_mask, vdb_128, vdb_128_mask


In [9]:
# check if names saved in text file is same
with open('test_names_file.txt', 'r') as f:
    saved_names = f.read().splitlines()

In [10]:
_dir = '/data/workspaces/spanwar/dataset/preprocessing_nmc_data/data_preprocessing/get_groundtruth_NMC/gt_large'
file = os.path.join(_dir, saved_names[5])  # Use the first saved name for testing
with h5py.File(file, 'r') as h5_file:
    sdf_32 = h5_file[f'{32}_sdf'][:]
    sdf_64 = h5_file[f'{64}_sdf'][:]
    sdf_128 = h5_file[f'{128}_sdf'][:]



In [11]:
vdb_32, vdb_64, _, vdb_128, vdb_128_d = exp('00000020.hdf5')  # Use the first saved name for testing

max sdf_32: 0.5854236 min sdf_32: -0.038727265
max sdf_64: 0.5854236 min sdf_64: -0.04543257
intial disable shape: torch.Size([0, 3])
subdivide_grid shape: torch.Size([22510, 3])
enabled_grid shape: torch.Size([22510, 3])
disabled_grid shape: torch.Size([0, 3])
large_vdb.jdata torch.Size([22510, 1])
up_feat.jdata torch.Size([22510, 1])
####################
large_vdb shape: torch.Size([22510, 1])
large_vdb_e shape: torch.Size([18681, 3])
sum of masks: 3829
intial disable shape: torch.Size([3829, 3])
subdivide_grid shape: torch.Size([197724, 3])
enabled_grid shape: torch.Size([125467, 3])
disabled_grid shape: torch.Size([72257, 3])
large_vdb.jdata torch.Size([125467, 1])
up_feat.jdata torch.Size([125467, 1])
####################
large_vdb shape: torch.Size([197724, 1])
large_vdb_e shape: torch.Size([98720, 3])
sum of masks: 26747


In [12]:
v, f, _ =vdb_32.grid.marching_cubes(vdb_32.data)
plot(v.jdata.detach().cpu().numpy(), f.jdata.detach().cpu().numpy())

feat = trilinear(vdb_128.grid, vdb_32)
vdb_128_tri = fvnn.VDBTensor(vdb_128.grid, feat)
v, f, _ =vdb_128_tri.grid.marching_cubes(vdb_128_tri.data)
plot(v.jdata.detach().cpu().numpy(), f.jdata.detach().cpu().numpy())

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.0, 0.00…

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.0, 0.00…

In [193]:
# vdb_64.grid.enable_ijk(vdb_64.grid.ijk.jdata)
# extract disable vdb
disabled_ijk = vdb_64.grid.ijk.rmask(vdb_64.grid.disabled_mask.jdata).jdata
disabled_mask = vdb_64.grid.ijk_to_index(disabled_ijk).jdata 
disabled_grid = fvdb.gridbatch_from_ijk(
    disabled_ijk,
    voxel_sizes=vdb_64.voxel_sizes,
    origins=vdb_64.origins,
    mutable=True
)
disabled_feats = vdb_64.jdata[disabled_mask]
disabled_grid_feats = disabled_grid.jagged_like(disabled_feats)
vdb_64_disabled = fvnn.VDBTensor(disabled_grid, disabled_grid_feats)

all_ijk = vdb_64_disabled.grid.ijk
m3g = torch.tensor(mt.mesh_grid(3), device=vdb_64_disabled.grid.device) - 1
fine = (2*all_ijk.jdata[:, None, :] + m3g[None, :, :]).view(-1, 3)
fine_grid = fvdb.gridbatch_from_ijk(
    fvdb.JaggedTensor(fine),
    voxel_sizes=vdb_64_disabled.grid.voxel_sizes / 2,
    origins=vdb_64_disabled.grid.origins,
    mutable=True
)
new_centers = fine_grid.grid_to_world(fine_grid.ijk.float())
up_feat = vdb_64_disabled.grid.sample_trilinear(new_centers, vdb_64_disabled.jdata)
fine_vdb_disabled = fvnn.VDBTensor(fine_grid, up_feat)

# extract enabled vdb
all_ijk = vdb_64.grid.ijk.jdata
fine_ijk = (2*all_ijk[:, None, :] + m3g[None, :, :]).view(-1, 3)
fine_grid = fvdb.gridbatch_from_ijk(
    fine_ijk,
    voxel_sizes=vdb_64.voxel_sizes/2,
    origins=vdb_64.origins,
    mutable=True
)
fine_grid.disable_ijk(fvdb.JaggedTensor([fine_vdb_disabled.grid.ijk.jdata]))
enabled_fine = fine_grid.ijk_enabled.jdata
enabled_grid = fvdb.gridbatch_from_ijk(
    enabled_fine,
    voxel_sizes=vdb_64.voxel_sizes/2,
    origins=vdb_64.origins,
    mutable=True
)
values = fetch_numpy_values(enabled_grid, sdf_128, 2*65-1)
values = scaled_sdf(values, 65)
enabled_vdb = fvnn.VDBTensor(enabled_grid, enabled_grid.jagged_like(values))

In [194]:
def find_matching_indices_string_based(tensor1, tensor2):
    """
    Convert coordinates to strings for exact matching
    Most reliable for any range of values
    """
    # Convert to string representations
    str_dict2 = {}
    for idx, row in enumerate(tensor2):
        key = f"{row[0].item()}_{row[1].item()}_{row[2].item()}"
        if key not in str_dict2:
            str_dict2[key] = []
        str_dict2[key].append(idx)
    
    matches1 = []
    matches2 = []
    
    for idx1, row in enumerate(tensor1):
        key = f"{row[0].item()}_{row[1].item()}_{row[2].item()}"
        if key in str_dict2:
            for idx2 in str_dict2[key]:
                matches1.append(idx1)
                matches2.append(idx2)
    
    return torch.tensor(matches2)

In [195]:
merged_coords = torch.cat([enabled_vdb.grid.ijk.jdata, fine_vdb_disabled.grid.ijk.jdata], dim=0)
merged_feats = torch.cat([enabled_vdb.jdata, fine_vdb_disabled.jdata], dim=0)


merged_grid = fvdb.gridbatch_from_ijk(
    fvdb.JaggedTensor(merged_coords),
    voxel_sizes=enabled_vdb.grid.voxel_sizes,
    origins=enabled_vdb.grid.origins,
    mutable=True
)
merged_ijk = merged_grid.ijk.jdata


# merged_feats = merged_feats[matches]
index = find_matching_indices_string_based(merged_ijk, merged_coords)
print(len(find_matching_indices_string_based(merged_coords, merged_ijk).unique()))
print(merged_ijk)
print(merged_coords[index]==merged_ijk)
print(merged_coords.shape)
print(merged_feats.shape)
merged_feats = merged_feats[index]
merged_vdb = fvnn.VDBTensor(merged_grid, merged_grid.jagged_like(merged_feats))
# ffg = fvnn.FillFromGrid()
# filled_disabled = ffg(fine_vdb_disabled, merged_grid)
# filled_enabled = ffg(enabled_vdb, merged_grid)
# merged_vdb = ffg(fine_vdb_disabled, filled_enabled)
# merged_vdb.grid.disable_ijk(fine_vdb_disabled.grid.ijk)
merged_vdb.grid.ijk.jdata.shape, merged_vdb.grid.ijk_enabled.jdata.shape

272055
tensor([[-35, -59, -19],
        [-35, -59, -18],
        [-35, -59, -17],
        ...,
        [ 27,  31,  25],
        [ 27,  31,  26],
        [ 27,  31,  27]], dtype=torch.int32)
tensor([[True, True, True],
        [True, True, True],
        [True, True, True],
        ...,
        [True, True, True],
        [True, True, True],
        [True, True, True]])
torch.Size([272055, 3])
torch.Size([272055, 1])


(torch.Size([272055, 3]), torch.Size([272055, 3]))

In [196]:
fine_vdb_disabled.grid.ijk.jdata.shape, enabled_vdb.grid.ijk.jdata.shape, merged_vdb.grid.ijk.jdata.shape

(torch.Size([147082, 3]), torch.Size([124973, 3]), torch.Size([272055, 3]))

In [197]:
(merged_vdb.jdata==0).sum()

tensor(0)

In [198]:
vdb_128.grid.enable_ijk(vdb_128.grid.ijk.jdata)
vdb_64.jdata.shape, vdb_64.grid.ijk_enabled.jdata.shape, vdb_64_disabled.jdata.shape, vdb_128.jdata.shape

(torch.Size([31351, 1]),
 torch.Size([18821, 3]),
 torch.Size([12530, 1]),
 torch.Size([272055, 1]))

In [200]:
# plot vbs using marching cube
nv, nf, _ = merged_vdb.grid.marching_cubes(merged_vdb.data)
nv = nv.jdata.numpy()
nf = nf.jdata.numpy()
# plot the mesh
plot(nv, nf)
# nv, nf

# plot vbs using marching cube
vdb_128.grid.enable_ijk(vdb_128.grid.ijk.jdata)
nv, nf, _ = vdb_64.grid.marching_cubes(vdb_64.data)
nv = nv.jdata.numpy()
nf = nf.jdata.numpy()
# plot the mesh
plot(nv, nf)

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(-1.490116…

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(-1.490116…